In [ ]:
#Install dependencies
%pip install anthropic python-dotenv

In [ ]:
#Load environment variables
from dotenv import load_dotenv

load_dotenv()

In [ ]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
#Make a request
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)  

def chat(messages, system=None, temperature=1.0, stop_sequence=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequence,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [ ]:
import json

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompt that generate Python, 
    JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires 
    Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
    {
        "task": "Description of task",
    },
    ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
"""

    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text=chat(messages, stop_sequence=["```"])
    return json.loads(text)

In [ ]:
dataset=generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
dataset

In [ ]:
# Function to take a test case and merges it with prompt template
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
# Function to orchestrates running a single test case and grading the result
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [ ]:
# Function to coordinate the entire evaluation process
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:
# Running the Evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
# Examining the result
print(json.dumps(results, indent=2))